In [1]:
import pandas as pd
import nltk
import re # pour les exprèssions régulières

In [2]:
# Téléchargement des ressources NLTK nécessaires
# 'punkt': pour découper le texte en mots (tokénisation)
# 'stopwords': liste de mots vides (la, le, de, des, the, is...)
# 'wordnet': dictionnaire pour la lemmatisation

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\MOISE\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\MOISE\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\MOISE\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\MOISE\AppData\Roaming\nltk_data...


True

In [4]:
# Chargement du dataset produit
df = pd.read_csv('../data/movie_processed.csv')

print("Shape:", df.shape)
df.head(2)

Shape: (4806, 3)


,movie_id,title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."


In [5]:
print(df['tags'][0])

In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy Science Fiction culture clash future space war space colony society space travel futuristic romance space alien tribe alien planet cgi marine soldier battle love affair anti war power relations mind and soul 3d Sam Worthington Zoe Saldana Sigourney Weaver James Cameron


In [13]:
# Nettoyage du dataset et normalisation
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Initialise le lemmatiseur
# La lemmatisation réduit un mot à sa forme de base
# "running" → "run", "movies" → "movie", "better" → "good"

lemmatizer = WordNetLemmatizer()

# Liste des mots vides en anglais (the, is, at, which...)
# Ces mots n'apportent aucune information sur le contenu du film
stop_words = set(stopwords.words('english'))

In [9]:
print(stop_words)

{'be', 'few', 'above', "they'd", 'ma', 'y', 'at', 'hasn', 'only', 'theirs', 'won', 'will', 'hadn', "it'd", "aren't", 'his', "it's", 'ourselves', 'once', 'what', 'ain', "they're", 'them', "shouldn't", 'before', "i've", 'shan', 'didn', "mightn't", 'a', 'when', 'why', "don't", 'your', 'yourselves', 'no', 'for', 'do', "she's", 'll', 'isn', "mustn't", "we'd", "he's", 'from', 'her', 'doing', "we're", 'nor', 'while', 'then', 'other', 'into', "you've", 'wasn', "shan't", 'myself', 'whom', "should've", 'my', "isn't", 'of', 'same', "they'll", 'again', "needn't", 'because', 'during', "they've", "i'll", 'not', "you'll", 'had', 'has', 'just', 'is', 'm', 'further', 'so', 'own', "couldn't", 'himself', 'both', 'weren', 'doesn', 'this', "didn't", 'him', 'as', 'itself', 'me', 'by', 'themselves', 'they', 'up', 's', 'and', 'can', 'haven', 'does', 'having', "he'll", 'their', 'its', 'who', "hasn't", "it'll", 'against', 'you', 'mightn', 're', 'very', 'those', 'wouldn', 'couldn', 'being', 'if', 'how', 'to', 'u

In [10]:
def clean_text(text):
    """
    Nettoie et normalise un texte pour le NLP.
    Etapes: minuscules --> suppression ponctuation -->
            tokenisation --> stopwords --> lemmatisation
    """
    # Etape 1: met tout en minuscules
    # "Action" et "action" deviennent identiques
    text = text.lower()

    # Etape 2: supprime tout ce qui n'est pas une lettre ou un espace
    # re.sub remplace les caractères spéciaux par un espace vide
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Etape 3: découpe le texte en liste de mots (tokenisation)
    # "action adventure fantasy" --> ["action", "adventure", "fantasy"]
    tokens = word_tokenize(text)

    # Etape 4: supprime les stopwords et lemmatise chaque mot
    tokens = [
        lemmatizer.lemmatize(word) # réduit le mot à sa forme de base
        for word in tokens
        if word not in stop_words and len(word) > 2 # garde uniquement les mots utiles ignore les courts (ex: "is", "at")
    ]

    # Etape 5: recolle les mots en une seule chaîne
    return " ".join(tokens)

In [11]:
# Test sur le premier film
print(clean_text(df['tags'][0]))

century paraplegic marine dispatched moon pandora unique mission becomes torn following order protecting alien civilization action adventure fantasy science fiction culture clash future space war space colony society space travel futuristic romance space alien tribe alien planet cgi marine soldier battle love affair anti war power relation mind soul sam worthington zoe saldana sigourney weaver james cameron


In [14]:
# On applique le nettoyage sur tout le dataset
df['tags_clean'] = df['tags'].apply(clean_text)

print("Nettoyage terminé !")
print(f"Nombre de films traités : {len(df)}")
df[['title', 'tags_clean']].head(3)

Nettoyage terminé !
Nombre de films traités : 4806


,title,tags_clean
0,Avatar,century paraplegic marine dispatched moon pand...
1,Pirates of the Caribbean: At World's End,captain barbossa long believed dead come back ...
2,Spectre,cryptic message bond past sends trail uncover ...


In [15]:
# Sauvegarde du dataset avec la nouvelle colonne tags_clean
df.to_csv('../data/movie_clean.csv', index=False)
print("Dataset sauvegardeé !")

Dataset sauvegardeé !
